# --- Day 14: Parabolic Reflector Dish ---
https://adventofcode.com/2023/day/14

You reach the place where all of the mirrors were pointing: a massive parabolic reflector dish attached to the side of another large mountain.

The dish is made up of many small mirrors, but while the mirrors themselves are roughly in the shape of a parabolic reflector dish, each individual mirror seems to be pointing in slightly the wrong direction. If the dish is meant to focus light, all it's doing right now is sending it in a vague direction.

This system must be what provides the energy for the lava! If you focus the reflector dish, maybe you can go where it's pointing and use the light to fix the lava production.

Upon closer inspection, the individual mirrors each appear to be connected via an elaborate system of ropes and pulleys to a large metal platform below the dish. The platform is covered in large rocks of various shapes. Depending on their position, the weight of the rocks deforms the platform, and the shape of the platform controls which ropes move and ultimately the focus of the dish.

In short: if you move the rocks, you can focus the dish. The platform even has a control panel on the side that lets you tilt it in one of four directions! The rounded rocks (O) will roll when the platform is tilted, while the cube-shaped rocks (#) will stay in place. You note the positions of all of the empty spaces (.) and rocks (your puzzle input). For example:
```

O....#....
O.OO#....#
.....##...
OO.#O....O
.O.....O#.
O.#..O.#.#
..O..#O..O
.......O..
#....###..
#OO..#....
```
Start by tilting the lever so all of the rocks will slide north as far as they will go:
```

OOOO.#.O..
OO..#....#
OO..O##..O
O..#.OO...
........#.
..#....#.#
..O..#.O.O
..O.......
#....###..
#....#....
```
You notice that the support beams along the north side of the platform are damaged; to ensure the platform doesn't collapse, you should calculate the total load on the north support beams.

The amount of load caused by a single rounded rock (O) is equal to the number of rows from the rock to the south edge of the platform, including the row the rock is on. (Cube-shaped rocks (#) don't contribute to load.) So, the amount of load caused by each rock in each row is as follows:
```

OOOO.#.O.. 10
OO..#....#  9
OO..O##..O  8
O..#.OO...  7
........#.  6
..#....#.#  5
..O..#.O.O  4
..O.......  3
#....###..  2
#....#....  1
```
The total load is the sum of the load caused by all of the rounded rocks. In this example, the total load is 136.

**Tilt the platform so that the rounded rocks all roll north. Afterward, what is the total load on the north support beams?**

In [81]:
import numpy as np
from collections import Counter

# hold the patterns
platform = []

# load data
with open("input.txt", "r") as f:
# with open("sample.txt", "r") as f:
    lines = f.read().splitlines()
    for line in lines:
        platform.append([x for x in line])

platform = np.array(platform)

# transpose it
tplatform = np.transpose(platform)

[rows,cols] = np.shape(tplatform)

def findNextOpenSpot(row, item_idx):
    tocheck = list(reversed(row[:item_idx]))
    out = item_idx
    for i in tocheck:
        if i == ".":
            out-=1
        else:
            break
    return out
        

for row_index,row in enumerate(tplatform):
    for item_idx, item in enumerate(row):
        if item_idx == 0:
            continue
        if item == "O":
            x = findNextOpenSpot(row, item_idx)
            if x != item_idx:
                tplatform[row_index][x] = "O"
                tplatform[row_index][item_idx] = "."

scores =[]
# transpose it back
platform = np.transpose(tplatform)
for i,v in enumerate(platform):
    # print(v,i, rows-i, Counter(v)["O"])
    Os = Counter(v)["O"]
    distance = rows-i
    score = Os*distance
    scores.append(score)

print(sum(scores))

107430


# --- Part Two ---
The parabolic reflector dish deforms, but not in a way that focuses the beam. To do that, you'll need to move the rocks to the edges of the platform. Fortunately, a button on the side of the control panel labeled "spin cycle" attempts to do just that!

Each cycle tilts the platform four times so that the rounded rocks roll north, then west, then south, then east. After each tilt, the rounded rocks roll as far as they can before the platform tilts in the next direction. After one cycle, the platform will have finished rolling the rounded rocks in those four directions in that order.

Here's what happens in the example above after each of the first few cycles:

After 1 cycle:
```

.....#....
....#...O#
...OO##...
.OO#......
.....OOO#.
.O#...O#.#
....O#....
......OOOO
#...O###..
#..OO#....
```
After 2 cycles:
```

.....#....
....#...O#
.....##...
..O#......
.....OOO#.
.O#...O#.#
....O#...O
.......OOO
#..OO###..
#.OOO#...O
```
After 3 cycles:
```

.....#....
....#...O#
.....##...
..O#......
.....OOO#.
.O#...O#.#
....O#...O
.......OOO
#...O###.O
#.OOO#...O
```
This process should work if you leave it running long enough, but you're still worried about the north support beams. To make sure they'll survive for a while, you need to calculate the total load on the north support beams after 1000000000 cycles.

In the above example, after 1000000000 cycles, the total load on the north support beams is 64.

**Run the spin cycle for 1000000000 cycles. Afterward, what is the total load on the north support beams?**

In [51]:
import numpy as np
from collections import Counter

# hold the patterns
platform = []

# load data
with open("input.txt", "r") as f:
# with open("sample.txt", "r") as f:
    lines = f.read().splitlines()
    for line in lines:
        platform.append([x for x in line])

originalPlatform = np.array(platform)

def findNextOpenSpot(row, item_idx):
    tocheck = list(reversed(row[:item_idx]))
    out = item_idx
    for i in tocheck:
        if i == ".":
            out-=1
        else:
            break
    return out

# get the state
def getPlatformState(gPlatform):
    for row_index,row in enumerate(gPlatform):
        for item_idx, item in enumerate(row):
            if item_idx == 0:
                continue
            if item == "O":
                x = findNextOpenSpot(row, item_idx)
                if x != item_idx:
                    gPlatform[row_index][x] = "O"
                    gPlatform[row_index][item_idx] = "."
    return gPlatform

# using numpy to flip this thing in the right direction
def runCycle(xplatform):
    """ 
    North = np.transpose(originalplatform)
    West = np.flipud(originalplatform)
    South = np.fliplr(np.transpose(originalplatform))
    East = np.fliplr(originalplatform)
    """
    directions = ['north','west','south','east']
    for i in directions:
        if i == 'north':
            xplatform = np.transpose(xplatform) # transpose
            xplatform = getPlatformState(xplatform)
            xplatform = np.transpose(xplatform) # transpose back
        elif i == 'west':
            xplatform = np.flipud(xplatform) # transpose
            xplatform = getPlatformState(xplatform)
            xplatform = np.flipud(xplatform) # transpose back
        elif i == 'south':
            xplatform = np.transpose(xplatform)
            xplatform = np.fliplr(xplatform) # transpose
            xplatform = getPlatformState(xplatform)
            xplatform = np.fliplr(xplatform) # transpose
            xplatform = np.transpose(xplatform)
        elif i == 'east':
            xplatform = np.fliplr(xplatform) # transpose
            xplatform = getPlatformState(xplatform)
            xplatform = np.fliplr(xplatform) # transpose back
        
    return xplatform

results =[]
cycles = 1000000000
newplatform = originalPlatform
for i in range(0,300):
    newplatform = runCycle(newplatform)
    # print('---')
    # print(newplatform)

    scores =[]
    # transpose it back
    # platform = np.transpose(tplatform)
    for i2,v in enumerate(newplatform):
        # print(v,i, rows-i, Counter(v)["O"])
        Os = Counter(v)["O"]
        distance = len(newplatform)-i2
        score = Os*distance
        scores.append(score)
    results.append(sum(scores))

# this is a kinda weird solution but it worked, don't ask me how
# so i know you cannot work with 1.000.000.000 cycles, so you have to find a repeating pattern
# so to find that i just run it a couple hundred times, keep track of most common scores and try one of the 
# most common ones, the first one was the answer xD
print(Counter(results))

Counter({96317: 24, 96297: 24, 96314: 23, 96325: 23, 96333: 23, 96344: 23, 96345: 23, 96340: 23, 96293: 23, 96420: 3, 96546: 2, 96518: 2, 96389: 2, 97959: 1, 97966: 1, 97962: 1, 97942: 1, 97920: 1, 97908: 1, 97873: 1, 97885: 1, 97866: 1, 97895: 1, 97820: 1, 97773: 1, 97747: 1, 97686: 1, 97615: 1, 97546: 1, 97499: 1, 97467: 1, 97447: 1, 97422: 1, 97390: 1, 97377: 1, 97368: 1, 97350: 1, 97317: 1, 97283: 1, 97235: 1, 97222: 1, 97181: 1, 97123: 1, 97083: 1, 97072: 1, 97047: 1, 97005: 1, 96961: 1, 96898: 1, 96863: 1, 96821: 1, 96762: 1, 96726: 1, 96688: 1, 96630: 1, 96617: 1, 96615: 1, 96578: 1, 96565: 1, 96520: 1, 96493: 1, 96459: 1, 96457: 1, 96439: 1, 96417: 1, 96415: 1, 96416: 1, 96369: 1, 96361: 1, 96347: 1, 96322: 1, 96310: 1, 96337: 1, 96351: 1, 96370: 1, 96392: 1, 96407: 1, 96441: 1, 96456: 1, 96480: 1, 96510: 1, 96529: 1, 96573: 1, 96567: 1, 96543: 1, 96526: 1, 96509: 1, 96485: 1, 96468: 1, 96444: 1, 96413: 1, 96357: 1, 96324: 1, 96299: 1, 96292: 1})
